# アウトカム再定義 Step 2：ARIS以外の「死が近い認識」記載の洗い出し

- 陽性群：unex24=0 かつ 全期間でARIS記載なし（初回判定で何らかの該当記載があったはずの群）
- 陰性群：unex24=1
- 窓：死亡前1〜30日（死亡当日は死亡確認記録の混入を避けるため除外）
- 段階1：文字n-gramの群間比較（患者単位）
- 段階2：臨床種語辞書（カテゴリA〜E）での照合
  - A 予後の明言／B 家族への説明／C 看取りの方針・体制 → 採用候補
  - D 臨死期の徴候 → 出力のみ。判定に使わない
  - E 導入時ACP（DNAR等）→ 原則除外。死亡前1〜7日に書き換えがあれば採用候補
- 段階3：KWIC抽出（語ごとに無作為20件、判定欄つき）
- DBは読み取りのみ

In [6]:
import os, re, sqlite3, unicodedata
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

DB_PATH = os.environ.get("HANA_DB", "/Users/muna/Hana_research/data/db/Hana_Research.db")
OUT_DIR = Path(os.environ.get("HANA_OUT", "/Users/muna/Hana_research/data/output/outcome_redef"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

WIN_MIN, WIN_MAX = 1, 30      # 死亡前日数の範囲（両端含む）
E_RECENT_DAYS = 7             # カテゴリE：この日数以内の書き換えを採用候補
CONTEXT_CHARS = 40
NEG_AFTER_CHARS = 12          # 否定語を探す範囲（語の直後）
NGRAM_MIN, NGRAM_MAX = 2, 6
NGRAM_MIN_POS = 5             # 陽性群での出現患者数の下限
KWIC_N = 20
RANDOM_SEED = 2026

con = sqlite3.connect(DB_PATH)
PID = "CAST(CAST({col} AS INTEGER) AS TEXT)"

def norm(t):
    return unicodedata.normalize("NFKC", t) if isinstance(t, str) else ""

## 1. 母集団・カルテ読み込み（Step 1と同じ処理）

In [7]:
pop = pd.read_sql(f"SELECT {PID.format(col='Patient_ID')} AS Patient_ID, Study_ID FROM unex_study", con)
orig = pd.read_sql(f"SELECT {PID.format(col='Patient_ID')} AS Patient_ID, unex24 FROM unexpected_death", con).drop_duplicates("Patient_ID")
ev = pd.read_sql(f"""SELECT {PID.format(col='Patient_ID')} AS Patient_ID, death_date FROM event
                     WHERE death_date IS NOT NULL AND TRIM(death_date) <> ''""", con).drop_duplicates("Patient_ID")
ev["death_dt"] = pd.to_datetime(ev.death_date, errors="coerce").dt.normalize()
pop = pop.merge(ev[["Patient_ID", "death_dt"]], on="Patient_ID", how="left").merge(orig, on="Patient_ID", how="left")

con.execute("DROP TABLE IF EXISTS temp.pop_ids")
con.execute("CREATE TEMP TABLE pop_ids (Patient_ID TEXT PRIMARY KEY)")
con.executemany("INSERT INTO pop_ids VALUES (?)", [(i,) for i in pop.Patient_ID])
k = pd.read_sql(f"""
    SELECT {PID.format(col='k.Patient_ID')} AS Patient_ID, k.record_no, k.visit_type, k.visit_datetime,
           CASE WHEN k.nurse_name IS NOT NULL AND TRIM(k.nurse_name) <> '' THEN 1 ELSE 0 END AS has_nurse_name,
           k.karte_text
    FROM karte k JOIN temp.pop_ids p ON {PID.format(col='k.Patient_ID')} = p.Patient_ID
""", con)

DT_RE = re.compile(r"(\d{4})/(\d{1,2})/(\d{1,2})")
def parse_date(s):
    m = DT_RE.search(s) if isinstance(s, str) else None
    if not m:
        return pd.NaT
    try:
        return pd.Timestamp(*map(int, m.groups()))
    except ValueError:
        return pd.NaT

k["visit_date"] = pd.to_datetime(k.visit_datetime.map(parse_date))
k = k.merge(pop[["Patient_ID", "Study_ID", "death_dt", "unex24"]], on="Patient_ID", how="left")
k["days_before_death"] = (k.death_dt - k.visit_date).dt.days
k = k[k.days_before_death >= 0].copy()
k["t"] = k.karte_text.map(norm)
print("死亡日以前の記録:", len(k), "/ 患者:", k.Patient_ID.nunique())

死亡日以前の記録: 115129 / 患者: 1276


## 2. 群分け（全期間のARIS有無で陽性群を定義）

In [8]:
ARIS_RE = re.compile(r"(?<![A-Za-z])A?RIS(?=care|[^A-Za-z]|$)|アリス", re.I)
aris_pts = set(k.loc[k.t.str.contains(ARIS_RE), "Patient_ID"])

pop["aris_ever"] = pop.Patient_ID.isin(aris_pts).astype(int)
pop["group"] = np.select(
    [(pop.unex24 == 0) & (pop.aris_ever == 0), pop.unex24 == 1],
    ["pos_nonaris", "neg_unex"], default="other")
display(pd.crosstab(pop.group, pop.aris_ever, margins=True))

win = k[(k.days_before_death >= WIN_MIN) & (k.days_before_death <= WIN_MAX)].merge(
    pop[["Patient_ID", "group"]], on="Patient_ID", how="left")
cov = win.groupby("group").Patient_ID.nunique()
print("窓内記録のある患者数（群別）:"); display(pd.concat([pop.group.value_counts().rename("n_total"), cov.rename("n_with_window_records")], axis=1))

aris_ever,0,1,All
group,,,
neg_unex,230,9,239
other,1,389,390
pos_nonaris,652,0,652
All,883,398,1281


窓内記録のある患者数（群別）:


,n_total,n_with_window_records
group,,
pos_nonaris,652,495
other,390,384
neg_unex,239,238


## 3. 段階1：文字n-gramの群間比較

- 1患者で複数回出ても1と数える
- 対数オッズ比は各セルに0.5を加えて計算
- 陽性群で5人以上に出現した語のみ

In [9]:
KEEP = re.compile(r"[ぁ-んァ-ヶ一-龥A-Za-z]")
def patient_grams(texts):
    t = "".join(texts)
    out = set()
    for n in range(NGRAM_MIN, NGRAM_MAX + 1):
        for i in range(len(t) - n + 1):
            g = t[i:i + n]
            if g.isspace() or any(c.isspace() for c in g) or not KEEP.search(g):
                continue
            out.add(g)
    return out

texts = win.groupby("Patient_ID").t.apply(list)
grp = pop.set_index("Patient_ID").group
cnt = {"pos_nonaris": Counter(), "neg_unex": Counter()}
n_pt = Counter()
for pid, tx in texts.items():
    g = grp.get(pid)
    if g in cnt:
        cnt[g].update(patient_grams(tx))
        n_pt[g] += 1

npos, nneg = n_pt["pos_nonaris"], n_pt["neg_unex"]
rows = [(g, c, cnt["neg_unex"].get(g, 0)) for g, c in cnt["pos_nonaris"].items() if c >= NGRAM_MIN_POS]
ng = pd.DataFrame(rows, columns=["ngram", "n_pos", "n_neg"])
ng["len"] = ng.ngram.str.len()
ng["pct_pos"] = ng.n_pos / npos * 100
ng["pct_neg"] = ng.n_neg / nneg * 100
a, b = ng.n_pos + .5, npos - ng.n_pos + .5
c, d = ng.n_neg + .5, nneg - ng.n_neg + .5
ng["log_or"] = np.log((a * d) / (b * c))
ng["pct_diff"] = ng.pct_pos - ng.pct_neg
ng = ng.sort_values(["log_or", "n_pos"], ascending=False)

print(f"陽性群 {npos}例 / 陰性群 {nneg}例 / 候補n-gram {len(ng)}")
ng.to_csv(OUT_DIR / "06_ngram_pos_vs_neg.csv", index=False, encoding="utf-8-sig")
display(ng[ng.n_pos >= 20].head(60))

陽性群 495例 / 陰性群 238例 / 候補n-gram 142264


,ngram,n_pos,n_neg,len,pct_pos,pct_neg,log_or,pct_diff
29110,I-20,40,0,4,8.080808,0.0,3.747423,8.080808
31436,II-2,40,0,4,8.080808,0.0,3.747423,8.080808
32827,II-20,39,0,5,7.878788,0.0,3.720228,7.878788
872,座薬を,37,0,3,7.474747,0.0,3.663897,7.474747
101937,II-30,34,0,5,6.868687,0.0,3.573994,6.868687
102011,II-3,34,0,4,6.868687,0.0,3.573994,6.868687
102031,I-30,34,0,4,6.868687,0.0,3.573994,6.868687
31257,に22,29,0,3,5.858586,0.0,3.406649,5.858586
38382,口摂取でき,28,0,5,5.656566,0.0,3.370021,5.656566
39772,経口摂取でき,28,0,6,5.656566,0.0,3.370021,5.656566


## 4. 段階2：臨床種語辞書

- 語は正規化（NFKC）後のテキストに対して照合
- `negated`：語の直後に否定語がある（目視確認用の目安。判定には使わない）
- 辞書は暫定。KWICの結果を見て追加・削除する

In [10]:
SEED = {
    "A_prognosis": {
        "余命": r"余命",
        "予後": r"予後",
        "時間〜月単位": r"(?:時間|日|週|月)単位",
        "厳しい": r"厳しい|厳しく",
        "長くない": r"長くない|長くは(?:ない|もたない)",
        "終末期": r"終末期|ターミナル|エンドオブライフ|EOL",
        "最期": r"最期|死期",
        "看取り期": r"看取り期|臨死期",
    },
    "B_family_explanation": {
        "家族へ説明": r"(?:家族|妻|夫|息子|娘|長男|長女|次男|次女|嫁|KP|キーパーソン).{0,20}(?:説明|IC|ムンテラ|お話|話し)",
        "覚悟": r"覚悟",
        "急変の可能性": r"急変(?:の可能性|もあり|もあ|しうる|し得る|のリスク)",
        "亡くなる": r"亡くな|死亡の可能性|死が近",
    },
    "C_eol_plan": {
        "看取り": r"看取り|看取る|お看取り",
        "死亡時の連絡": r"(?:死亡|呼吸停止|心停止).{0,10}(?:時|際).{0,10}(?:連絡|対応)",
        "エンゼルケア": r"エンゼル",
    },
    "D_dying_signs": {
        "下顎呼吸": r"下顎呼吸",
        "死前喘鳴": r"死前喘鳴",
        "尿量低下": r"尿量(?:低下|減少)|無尿",
        "チアノーゼ": r"チアノーゼ",
    },
    "E_acp": {
        "DNAR": r"(?<![A-Za-z])DNA?R(?![A-Za-z])",
        "延命": r"延命",
        "蘇生": r"蘇生|CPR|心マ",
        "挿管・人工呼吸器": r"挿管|人工呼吸器",
        "搬送しない": r"(?:搬送|救急車).{0,10}(?:しない|希望(?:せず|なし|しない)|呼ばない)",
    },
}
NEG_RE = re.compile(r"なし|無し|ない|せず|しない|否定|希望されず")

rows = []
for r in k.itertuples(index=False):   # Eの書き換え判定に全期間が必要なため全記録を走査
    for cat, terms in SEED.items():
        for term, pat in terms.items():
            for m in re.finditer(pat, r.t, flags=re.I):
                s, e = m.start(), m.end()
                rows.append({
                    "Study_ID": r.Study_ID, "Patient_ID": r.Patient_ID, "unex24": r.unex24,
                    "record_no": r.record_no, "visit_datetime": r.visit_datetime,
                    "days_before_death": r.days_before_death,
                    "visit_type": r.visit_type, "has_nurse_name": r.has_nurse_name,
                    "category": cat, "term": term, "matched": m.group(0),
                    "negated": int(bool(NEG_RE.search(r.t[e:e + NEG_AFTER_CHARS]))),
                    "context": r.t[max(0, s - CONTEXT_CHARS): e + CONTEXT_CHARS].replace("\n", " / "),
                })
seed = pd.DataFrame(rows).merge(pop[["Patient_ID", "group"]], on="Patient_ID", how="left")
seed["in_window"] = seed.days_before_death.between(WIN_MIN, WIN_MAX).astype(int)
print("種語一致（全期間）:", len(seed), "/ 窓内:", seed.in_window.sum())

種語一致（全期間）: 61352 / 窓内: 10183


### カテゴリEの書き換え判定

- `e_first_recent`：その語が死亡前1〜7日に**初めて**出現
- `e_changed_recent`：死亡前1〜7日の記載の文脈（前後40字）が、それ以前のどの記載とも一致しない
- テンプレートの毎回コピーは `e_changed_recent=0` になる想定

In [11]:
E = seed[seed.category == "E_acp"].sort_values(["Patient_ID", "term", "days_before_death"], ascending=[True, True, False]).copy()
E_recent = E.days_before_death.between(WIN_MIN, E_RECENT_DAYS)

first_day = E.groupby(["Patient_ID", "term"]).days_before_death.transform("max")
E["e_first_recent"] = (E_recent & (first_day <= E_RECENT_DAYS)).astype(int)

prev_ctx = {}
changed = []
for r in E.itertuples():
    key = (r.Patient_ID, r.term)
    seen = prev_ctx.setdefault(key, set())
    is_recent = WIN_MIN <= r.days_before_death <= E_RECENT_DAYS
    changed.append(int(is_recent and r.context not in seen and len(seen) > 0))
    if r.days_before_death > E_RECENT_DAYS:
        seen.add(r.context)
E["e_changed_recent"] = changed
E["e_recent_adopt_candidate"] = ((E.e_first_recent == 1) | (E.e_changed_recent == 1)).astype(int)

seed = seed.merge(E[["record_no", "term", "context", "e_first_recent", "e_changed_recent", "e_recent_adopt_candidate"]]
                  .drop_duplicates(), on=["record_no", "term", "context"], how="left")
for c in ["e_first_recent", "e_changed_recent", "e_recent_adopt_candidate"]:
    seed[c] = seed[c].fillna(0).astype(int)

seed.to_csv(OUT_DIR / "07_seed_hits_long.csv", index=False, encoding="utf-8-sig")
print("E 採用候補の記録数:", E.e_recent_adopt_candidate.sum(),
      "/ 患者数:", E.loc[E.e_recent_adopt_candidate == 1, "Patient_ID"].nunique())

E 採用候補の記録数: 380 / 患者数: 202


## 5. 群別の出現状況（窓内、患者単位）

In [12]:
sw = seed[(seed.in_window == 1) & (seed.negated == 0)]
sw = sw[(sw.category != "E_acp") | (sw.e_recent_adopt_candidate == 1)]

n_grp = win.groupby("group").Patient_ID.nunique()
tab = (sw.groupby(["category", "term", "group"]).Patient_ID.nunique().unstack("group").fillna(0).astype(int))
for g in ["pos_nonaris", "neg_unex"]:
    if g not in tab: tab[g] = 0
    tab[f"pct_{g}"] = (tab[g] / n_grp.get(g, np.nan) * 100).round(1)
display(tab)

cat_pt = (sw[sw.category != "D_dying_signs"].groupby(["Patient_ID", "category"]).size().unstack(fill_value=0) > 0).astype(int)
summ = pop.set_index("Patient_ID")[["Study_ID", "unex24", "aris_ever", "group"]].join(cat_pt).fillna(0)
cats = [c for c in ["A_prognosis", "B_family_explanation", "C_eol_plan", "E_acp"] if c in summ]
summ["any_ABCE"] = (summ[cats].sum(axis=1) > 0).astype(int)
summ["any_ABCE_or_aris"] = ((summ.any_ABCE == 1) | (summ.aris_ever == 1)).astype(int)
print("初回判定 × 種語(A/B/C/E)あり")
display(pd.crosstab(summ.unex24, summ.any_ABCE, margins=True))
print("初回判定 × (種語 または ARIS)あり")
display(pd.crosstab(summ.unex24, summ.any_ABCE_or_aris, margins=True))
summ.reset_index().to_csv(OUT_DIR / "08_seed_patient_summary.csv", index=False, encoding="utf-8-sig")

group                          neg_unex  other  pos_nonaris  pct_pos_nonaris  \
category             term                                                      
A_prognosis          予後               5     25           25              5.1   
                     余命               0     10            3              0.6   
                     厳しい              8     50           42              8.5   
                     時間〜月単位           2      6            4              0.8   
                     最期              11    109           60             12.1   
                     看取り期             0      1            0              0.0   
                     終末期             74    235          189             38.2   
                     長くない             0      7            5              1.0   
B_family_explanation 亡くなる             1     20           19              3.8   
                     家族へ説明          117    306          360             72.7   
                     急変の可能性           5     41           26              5.3   
                     覚悟               2     39           10              2.0   
C_eol_plan           エンゼルケア           0      3            0              0.0   
                     死亡時の連絡           0      0            2              0.4   
                     看取り             30    257          141             28.5   
D_dying_signs        チアノーゼ            9     41           48              9.7   
                     下顎呼吸            16     77           46              9.3   
                     尿量低下             7     30           22              4.4   
E_acp                DNAR             0      3            8              1.6   
                     延命               0     27           19              3.8   
                     挿管・人工呼吸器         1      6           12              2.4   
                     搬送しない           13     36           15              3.0   
                     蘇生               3      7            7              1.4   

group                          pct_neg_unex  
category             term                    
A_prognosis          予後                 2.1  
                     余命                 0.0  
                     厳しい                3.4  
                     時間〜月単位             0.8  
                     最期                 4.6  
                     看取り期               0.0  
                     終末期               31.1  
                     長くない               0.0  
B_family_explanation 亡くなる               0.4  
                     家族へ説明             49.2  
                     急変の可能性             2.1  
                     覚悟                 0.8  
C_eol_plan           エンゼルケア             0.0  
                     死亡時の連絡             0.0  
                     看取り               12.6  
D_dying_signs        チアノーゼ              3.8  
                     下顎呼吸               6.7  
                     尿量低下               2.9  
E_acp                DNAR               0.0  
                     延命                 0.0  
                     挿管・人工呼吸器           0.4  
                     搬送しない              5.5  
                     蘇生                 1.3

初回判定 × 種語(A/B/C/E)あり


any_ABCE,0,1,All
unex24,,,
0.0,236,806,1042
1.0,73,166,239
All,309,972,1281


初回判定 × (種語 または ARIS)あり


any_ABCE_or_aris,0,1,All
unex24,,,
0.0,223,819,1042
1.0,72,167,239
All,295,986,1281


## 6. 段階3：KWIC（語ごとに無作為20件）

- 窓内・全群から抽出。`group` 列で陽性・陰性どちらの記載か分かる
- `判定` 列に 採用／不採用／条件付き、`メモ` 列に理由を記入する
- 乱数シード固定（再現可能）

In [13]:
kw = seed[seed.in_window == 1]
kwic = (kw.sample(frac=1, random_state=RANDOM_SEED)
          .groupby(["category", "term"]).head(KWIC_N)
          .sort_values(["category", "term"]))
kwic = kwic[["category", "term", "group", "Study_ID", "days_before_death", "visit_type",
             "has_nurse_name", "negated", "matched", "context"]].copy()
kwic["判定"] = ""
kwic["メモ"] = ""
kwic.to_csv(OUT_DIR / "09_kwic_seed_review.csv", index=False, encoding="utf-8-sig")
print("KWIC行数:", len(kwic))
display(kwic.groupby(["category", "term"]).size())

KWIC行数: 406


category              term    
A_prognosis           予後          20
                      余命          20
                      厳しい         20
                      時間〜月単位      20
                      最期          20
                      看取り期         1
                      終末期         20
                      長くない        17
B_family_explanation  亡くなる        20
                      家族へ説明       20
                      急変の可能性      20
                      覚悟          20
C_eol_plan            エンゼルケア       3
                      死亡時の連絡       4
                      看取り         20
D_dying_signs         チアノーゼ       20
                      下顎呼吸        20
                      尿量低下        20
                      死前喘鳴         1
E_acp                 DNAR        20
                      延命          20
                      挿管・人工呼吸器    20
                      搬送しない       20
                      蘇生          20
dtype: int64

In [14]:
c = pd.read_csv(OUT / "2_changed_ids.csv", dtype={"Patient_ID": str})

def change(row, col):
    if row["in_old"] == row[col]:
        return "変化なし"
    return "旧のみ（抜けた）" if row["in_old"] == 1 else "新のみ（増えた）"

c["chg_A"] = c.apply(change, axis=1, col="in_new_A")
c["chg_B"] = c.apply(change, axis=1, col="in_new_B")

print("=== A ===")
print(pd.crosstab(c["reason_or_status"], c["chg_A"]))
print("\n=== B ===")
print(pd.crosstab(c["reason_or_status"], c["chg_B"]))

# 期間外557例のうち First_Visit_Date が空の数
print("\n初診日なし:", pm["fv"].isna().sum(),
      " / 初診日あり・期間外:", (pm["fv"].notna() & pm["r_fv"]).sum())

NameError: name 'OUT' is not defined

In [ ]:
con.close()